# QuranMB.v2 ASR Evaluation — Qwen3-ASR-1.7B via **vLLM backend**

Evaluates `Qwen/Qwen3-ASR-1.7B` on `IqraEval/QuranMB.v2` (test split) using phoneme-level PER and BLEU.

This is the vLLM-backend counterpart of the HF/`transformers`-backend notebook: same dataset,
same phonemizer, same metrics — only the model loading + inference calls change (`Qwen3ASRModel.LLM(...)`
instead of `Qwen3ASRModel.from_pretrained(...)`).


In [1]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — system packages
# ════════════════════════════════════════════════════════════════════
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1

# ════════════════════════════════════════════════════════════════════
# CELL 2 — qwen-asr with the vLLM extra, plus dataset/eval/phonemizer deps
# ════════════════════════════════════════════════════════════════════
!pip install -U "qwen-asr[vllm]" -q
!pip install -U jiwer sacrebleu datasets pandas mishkal soundfile huggingface_hub -q

# pin huggingface-hub back down to what the installed transformers expects
!pip install -U "huggingface-hub<1.0,>=0.34.0" --no-deps -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0

In [2]:
# ════════════════════════════════════════════════════════════════════
# CELL 3 — verify install
# ════════════════════════════════════════════════════════════════════
import torch, vllm, qwen_asr
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("vllm:", vllm.__version__)
print("qwen_asr:", qwen_asr.__version__ if hasattr(qwen_asr, "__version__") else "installed")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: No GPU detected — vLLM backend needs a CUDA GPU. "
          "Runtime → Change runtime type → GPU.")


ERROR 07-13 17:41:10 [fa_utils.py:82] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
torch: 2.9.1+cu128 | CUDA available: True
vllm: 0.14.0
qwen_asr: installed


In [3]:
# ════════════════════════════════════════════════════════════════════
# CELL 4 — HuggingFace token (needed to pull IqraEval + Qwen weights
# if they are gated/rate-limited)
# ════════════════════════════════════════════════════════════════════
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("WARNING: HF_TOKEN not found/not on Colab — set it manually via "
          "os.environ['HF_TOKEN'] = '...' if you hit auth errors.")

COLAB_WORKING = "/content" if os.path.isdir("/content") else "/tmp"


In [4]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — phonemizer setup (identical to the HF-backend notebook)
# ════════════════════════════════════════════════════════════════════
import sys, re, gc, tempfile, warnings, subprocess

PHONETISER_DIR = os.path.join(COLAB_WORKING, "MSA_phonetiser")
if not os.path.exists(PHONETISER_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/Iqra-Eval/MSA_phonetiser.git", PHONETISER_DIR],
        check=True,
    )

os.chdir(PHONETISER_DIR)
sys.path.insert(0, os.getcwd())

from phonetiser.phonetise_Arabic import phonetise
from mishkal.tashkeel import TashkeelClass

tashkeel = TashkeelClass()

def arabic_to_phonemes(arabic_text: str) -> str:
    diacritised = tashkeel.tashkeel(arabic_text)
    result = phonetise(diacritised)
    if not result or not result[1]:
        return ""
    phonemes = result[1][0]
    phonemes = phonemes.replace(" sil", "").replace("sil", "").strip()
    phonemes = re.sub(r'(\w+)[01]', r'\1', phonemes)
    return phonemes

# sanity check
print(arabic_to_phonemes("\u0625\u0650\u0646\u0651\u064e \u0627\u0644\u0644\u0651\u064e\u0647\u064e \u0628\u0650\u0643\u064f\u0644\u0651\u0650 \u0634\u064e\u064a\u0652\u0621\u064d \u0639\u064e\u0644\u0650\u064a\u0645\u064c"))


phoetising utterance
 <iion~a All~aha bikul~i $ayo'K EaliymN
['<', 'i0', 'nn', 'a']
['ll', 'a', 'h', 'a']
['b', 'i0', 'k', 'u0', 'll', 'i0']
['$', 'a', 'y', '<', 'i1', 'n']
['E', 'a', 'l', 'ii0', 'm', 'u1', 'n']
< i nn a ll a h a b i k u ll i $ a y < i n E a l ii m u n


In [5]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 — analysis imports + CONFIG
# ════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import soundfile as sf
import jiwer
import sacrebleu
import unicodedata
import io

warnings.filterwarnings("ignore")

DATASET_ID  = "IqraEval/QuranMB.v2"
REF_DATASET = "IqraEval/test_references"
SPLIT       = "test"
NUM_SAMPLES = 5          # set to 1640 for the full test set
TARGET_SR   = 16_000
MODEL_ID    = "Qwen/Qwen3-ASR-1.7B"

print(f"Device: {DEVICE}")
print(f"Model : {MODEL_ID} (vLLM backend)")


Device: cuda
Model : Qwen/Qwen3-ASR-1.7B (vLLM backend)


In [6]:
# ════════════════════════════════════════════════════════════════════
# CELL 7 — load dataset (streaming, undecoded audio) + references
# ════════════════════════════════════════════════════════════════════
from datasets import load_dataset, Audio

print(f"[INFO] Loading dataset: {DATASET_ID} / {SPLIT}")

dataset = load_dataset(DATASET_ID, split="test", streaming=True)
dataset = dataset.cast_column("audio", Audio(decode=False))  # prevent auto-decoding
samples = list(dataset.take(NUM_SAMPLES))

ref_dataset = load_dataset(REF_DATASET, split="test")
ref_lookup  = {row["ID"]: row["Reference_phn"] for row in ref_dataset}

print(f"Loaded {len(samples)} audio samples, {len(ref_lookup)} reference entries")


[INFO] Loading dataset: IqraEval/QuranMB.v2 / test


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/307 [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/21.1k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1642 [00:00<?, ? examples/s]

Loaded 5 audio samples, 1642 reference entries


In [7]:
# ════════════════════════════════════════════════════════════════════
# CELL 8 — metrics + text normalisation helpers (identical to HF notebook)
# ════════════════════════════════════════════════════════════════════
def normalise_arabic(text: str) -> str:
    """Strip diacritics and normalise Arabic text before scoring."""
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[\u0625\u0623\u0622\u0671]", "\u0627", text)
    text = text.replace("\u0640", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_per(references, hypotheses):
    return jiwer.wer(references, hypotheses)


def compute_bleu(references, hypotheses):
    return sacrebleu.corpus_bleu(hypotheses, [references], tokenize="13a")


In [8]:
# ════════════════════════════════════════════════════════════════════
# CELL 9 — load Qwen3-ASR-1.7B on the vLLM backend
#
# Qwen3ASRModel.LLM(...) is qwen-asr's vLLM-backed wrapper (as opposed
# to Qwen3ASRModel.from_pretrained(...), which is the transformers
# backend used in the original notebook). Same .transcribe() call
# signature either way, so the rest of the pipeline barely changes.
#
# On Linux (Colab included), multiprocessing defaults to "fork", so the
# `if __name__ == "__main__":` guard vLLM's own README shows for plain
# .py scripts generally isn't needed here — but if you see a spawn-related
# crash, restart the kernel and re-run this cell first, in isolation.
#
# NOTE: the 1.7B model is ~3x the 0.6B model. A fixed
# gpu_memory_utilization=0.85 is safe on a mostly-empty GPU but on a
# 16 GB T4 it can OOM (or hit a stale-distributed-state error on retries
# within the same kernel). The cleanup + dynamic sizing below make this
# cell safe to re-run without a kernel restart.
# ════════════════════════════════════════════════════════════════════
from qwen_asr import Qwen3ASRModel
import gc, torch

print(f"[LOAD] {MODEL_ID} (vLLM backend) on {DEVICE}")

# Free memory from any earlier failed load attempt in this session — vLLM builds
# weights/KV-cache/compiled graphs on the GPU before it raises, and since the
# process was never restarted between retries, that memory stays pinned.
if "asr_model" in dir():
    del asr_model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Tear down any leftover torch.distributed / vLLM parallel-state singletons from
# a previous failed Qwen3ASRModel.LLM() call in this same kernel. vLLM's
# parallel_state module keeps module-level globals (_WORLD, _TP, ...) pointing at
# the old ProcessGroup object; if it's stale or was never registered in torch's
# pg_map, the next init raises "Process group ... is not initialized in the world
# group map." Destroying both clears that stale state so init can run clean.
try:
    from vllm.distributed.parallel_state import (
        destroy_model_parallel,
        destroy_distributed_environment,
    )
    destroy_model_parallel()
    destroy_distributed_environment()
except Exception as e:
    print(f"[cleanup] no prior vLLM parallel state to destroy ({e})")

import torch.distributed as dist
if dist.is_initialized():
    dist.destroy_process_group()

# Reduces allocator fragmentation (must be set before any CUDA context exists to
# fully apply — harmless to set here regardless).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gb, total_gb = free_bytes / 1e9, total_bytes / 1e9
print(f"Free GPU memory before load: {free_gb:.2f} GiB / {total_gb:.2f} GiB total")

# If a meaningful chunk of the GPU is already occupied at this point, it's
# almost certainly unreleased memory from a prior failed load attempt in this
# same kernel. Restarting the runtime is the only fully reliable fix for that.
if free_gb < 0.7 * total_gb:
    print(
        f"[WARNING] Only {free_gb:.2f} GiB free out of {total_gb:.2f} GiB total. "
        "This strongly suggests memory from a previous failed vLLM load is still "
        "pinned on the GPU. Restart the runtime (Runtime > Restart session) and "
        "re-run from the top before continuing, or this load will likely OOM again."
    )

safety_margin_gb = 1.5  # headroom for KV-cache profiling on small GPUs
target_util = max(0.1, min(0.85, (free_gb - safety_margin_gb) / total_gb))
print(f"-> gpu_memory_utilization={target_util:.2f}")

asr_model = Qwen3ASRModel.LLM(
    model=MODEL_ID,
    gpu_memory_utilization=target_util,
    max_model_len=32768,
    max_inference_batch_size=8,    # smaller dummy multimodal profiling batch keeps
                                    # peak memory down during KV-cache sizing; 8 is
                                    # plenty for a small eval run
    max_new_tokens=256,
    enforce_eager=True,            # skips CUDA-graph capture — a memory-hungry,
                                    # throughput-only optimization not needed here
)

print("Model loaded.")


[LOAD] Qwen/Qwen3-ASR-1.7B (vLLM backend) on cuda
Free GPU memory before load: 15.53 GiB / 15.64 GiB total
-> gpu_memory_utilization=0.85
INFO 07-13 17:41:27 [utils.py:263] non-default args: {'max_model_len': 32768, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3-ASR-1.7B'}


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

INFO 07-13 17:41:28 [model.py:530] Resolved architecture: Qwen3ASRForConditionalGeneration


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


WARNING 07-13 17:41:32 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 07-13 17:47:01 [llm.py:347] Supported tasks: ['generate', 'transcription']
Model loaded.


In [9]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 — decode audio bytes -> (np.ndarray, sr) tuples for every sample
# ════════════════════════════════════════════════════════════════════
audio_inputs, sample_ids, ref_phonemes_list = [], [], []

for sample in samples:
    audio_bytes = sample["audio"]["bytes"]
    audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))
    sample_id = sample["ID"]

    audio_inputs.append((audio_array, sampling_rate))
    sample_ids.append(sample_id)
    ref_phonemes_list.append(ref_lookup.get(sample_id, ""))

print(f"Prepared {len(audio_inputs)} audio inputs for batched vLLM inference")


Prepared 5 audio inputs for batched vLLM inference


In [10]:
# ════════════════════════════════════════════════════════════════════
# CELL 11 — run inference (single batched call — vLLM handles the
# concurrency/scheduling internally, this is where the speedup vs. the
# transformers backend actually comes from)
# ════════════════════════════════════════════════════════════════════
results = asr_model.transcribe(
    audio=audio_inputs,
    language=["Arabic"] * len(audio_inputs),
    return_time_stamps=False,
)

hyp_phonemes_list = []
for idx, (r, sample_id) in enumerate(zip(results, sample_ids)):
    hyp_text = (r.text or "").strip()
    hyp_text = normalise_arabic(hyp_text)
    try:
        hyp_phonemes = arabic_to_phonemes(hyp_text)
    except Exception as e:
        print(f"  [{idx}] ID={sample_id} PHONEMIZE ERROR: {e}")
        hyp_phonemes = ""
    hyp_phonemes_list.append(hyp_phonemes)

    print(f"\n  [{idx}] ID: {sample_id}")
    print(f"      REF: {ref_phonemes_list[idx][:60]}...")
    print(f"      HYP: {hyp_phonemes[:60]}...")


phoetising utterance
 AnmA yaxo$aY Allhu min EibaAdihi AloEulamaA'u
0
1
['n', 'm', 'aa']
['n', 'm', 'a']
0
1
['y', 'a', 'x', '$', 'aa']
['y', 'a', 'x', '$', 'a']
['l', 'l', 'h', 'u0']
['m', 'i0', 'n']
['E', 'i0', 'b', 'aa', 'd', 'i0', 'h', 'i0']
['l', 'E', 'u0', 'l', 'a', 'm', 'aa', '<', 'u0']

  [0] ID: 00000_00001
      REF: < i nn a m aa y a x $ aa ll a h a m i n E i b aa d i h i l E...
      HYP: n m aa y a x $ aa l l h u m i n E i b aa d i h i l E u l a m...
phoetising utterance
 An~a Allha bariy'a min Alomu$orikiyna warasuwlihi
['nn', 'a']
skipped
['nn', 'a']
['l', 'l', 'h', 'a']
['b', 'a', 'r', 'ii0', '<', 'a']
['m', 'i0', 'n']
['l', 'm', 'u0', '$', 'r', 'i0', 'k', 'ii0', 'n', 'a']
['w', 'a', 'r', 'a', 's', 'uu0', 'l', 'i0', 'h', 'i0']

  [1] ID: 00000_00002
      REF: < a nn a ll a h a b a r ii < u n mm i n a l m u $ r i k ii n...
      HYP: nn a l l h a b a r ii < a m i n l m u $ r i k ii n a w a r a...
phoetising utterance
 waya*ub~u tal~aAy AbrAhym rab~ihi
['w', 'a', 'y',

In [11]:
# ════════════════════════════════════════════════════════════════════
# CELL 12 — corpus-level PER / BLEU
# ════════════════════════════════════════════════════════════════════
per  = compute_per(ref_phonemes_list, hyp_phonemes_list)
bleu = compute_bleu(ref_phonemes_list, hyp_phonemes_list)

print("=" * 70)
print(f"MODEL: {MODEL_ID} (vLLM backend)")
print("=" * 70)
print(f"Corpus PER  : {per * 100:.2f}%")
print(f"Corpus BLEU : {bleu.score:.2f}")

summary_df = pd.DataFrame([{
    "Model": MODEL_ID,
    "Backend": "vLLM",
    "PER (%)": round(per * 100, 2),
    "BLEU": round(bleu.score, 2),
}])
display(summary_df)


MODEL: Qwen/Qwen3-ASR-1.7B (vLLM backend)
Corpus PER  : 27.21%
Corpus BLEU : 58.56


,Model,Backend,PER (%),BLEU
0,Qwen/Qwen3-ASR-1.7B,vLLM,27.21,58.56


In [12]:
# ════════════════════════════════════════════════════════════════════
# CELL 13 — per-sample PER breakdown
# ════════════════════════════════════════════════════════════════════
for i, (ref, hyp) in enumerate(zip(ref_phonemes_list, hyp_phonemes_list)):
    p = compute_per([ref], [hyp])
    print(f"  [{i}] ID={sample_ids[i]}  PER={p*100:.1f}%")

print("\nDone — vLLM backend, designed for Colab T4/A100 (or any single CUDA GPU).")


  [0] ID=00000_00001  PER=20.0%
  [1] ID=00000_00002  PER=25.6%
  [2] ID=00000_00003  PER=42.3%
  [3] ID=00000_00004  PER=33.3%
  [4] ID=00000_00004439  PER=16.7%

Done — vLLM backend, designed for Colab T4/A100 (or any single CUDA GPU).
